# Small GPT-Style Language Model for Amharic (አማርኛ)
### Complete From-Scratch NumPy Implementation & End-to-End Walkthrough

This notebook demonstrates the end-to-end capabilities of **Amharic GPT**, built purely in **Python and NumPy** without deep learning frameworks (no PyTorch, TensorFlow, Transformers, or SentencePiece).

**Table of Contents:**
1. Setup & Environment
2. Amharic Tokenization (Character-level & BPE Subword)
3. Causal Multi-Head Self-Attention & Heatmap Visualization
4. Training & Convergence Loss Curves
5. Autoregressive Text Generation with KV Caching

## 1. Setup & Environment
Ensure project root is on `sys.path` and import pure NumPy modules.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Add repository root
repo_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.config import GPTConfig
from src.tokenizer import AmharicTokenizer
from src.bpe_tokenizer import BPETokenizer
from src.model import GPTLanguageModel
from src.utils import load_checkpoint
from src.generate import generate

print(f"NumPy Version: {np.__version__}")
print("Environment initialized successfully!")

## 2. Amharic Tokenization
Comparing character-level Fidel tokenization and Byte-Pair Encoding (BPE) subword tokenization.

In [ ]:
sample_text = "ሰላም ለኢትዮጵያ ሀገራችን ይሁን።"

# 1. Character-level Tokenizer
char_tokenizer = AmharicTokenizer()
char_tokenizer.build_vocab(sample_text)
char_tokens = char_tokenizer.encode(sample_text)
char_decoded = char_tokenizer.decode(char_tokens, skip_special_tokens=True)

print("=== Character-Level Tokenizer ===")
print(f"Original text: {sample_text}")
print(f"Encoded IDs ({len(char_tokens)} tokens): {char_tokens}")
print(f"Decoded text:  {char_decoded}")
assert char_decoded == sample_text, "Lossless roundtrip failed!"

# 2. Subword BPE Tokenizer
bpe_tokenizer = BPETokenizer()
bpe_tokenizer.train(sample_text * 10, target_vocab_size=35, verbose=False)
bpe_tokens = bpe_tokenizer.encode(sample_text)
bpe_decoded = bpe_tokenizer.decode(bpe_tokens)

print("\n=== BPE Subword Tokenizer ===")
print(f"Encoded IDs ({len(bpe_tokens)} tokens): {bpe_tokens}")
print(f"Decoded text:  {bpe_decoded}")
assert bpe_decoded == sample_text, "BPE lossless roundtrip failed!"

## 3. Causal Attention Weights & Triangular Heatmap
Visualizing the causal mask in action: past tokens can attend to prior and current positions, while future positions receive strictly 0 attention.

In [ ]:
# Initialize a tiny model to extract internal attention matrices
config = GPTConfig.get_tiny_config(vocab_size=char_tokenizer.vocab_size)
model = GPTLanguageModel(config, rng=np.random.RandomState(42))
model.eval()

input_ids = np.array([char_tokens[:8]], dtype=np.int64)
token_chars = [char_tokenizer.id_to_char.get(tid, "?") for tid in input_ids[0]]

# Forward pass
_ = model.forward(input_ids)

# Extract attention weights from Layer 0, Head 0
# Shape: (B, H, T, T) -> (T, T)
attn_weights = model.blocks[0].attn.cached_weights[0, 0, :, :]

# Plotting the attention heatmap
plt.figure(figsize=(7, 6))
plt.imshow(attn_weights, cmap="Blues", interpolation="nearest")
plt.colorbar(label="Attention Probability")
plt.xticks(range(len(token_chars)), token_chars, fontsize=12)
plt.yticks(range(len(token_chars)), token_chars, fontsize=12)
plt.xlabel("Key Position (Attended Tokens)", fontsize=11)
plt.ylabel("Query Position (Current Token)", fontsize=11)
plt.title("Layer 0 Head 0 Causal Self-Attention Heatmap\n(Upper-triangle strictly zeroed out)", fontsize=12)
plt.tight_layout()
plt.show()

## 4. Training Convergence & Loss Curves
Displays training vs validation loss curves across training iterations.

In [ ]:
metrics_file = os.path.join(repo_root, "outputs", "small_amharic_run", "metrics.csv")

if os.path.exists(metrics_file):
    import csv
    train_steps, train_loss = [], []
    val_steps, val_loss = [], []
    with open(metrics_file, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            s = int(row["step"])
            if row.get("train_loss"):
                train_steps.append(s)
                train_loss.append(float(row["train_loss"]))
            if row.get("val_loss"):
                val_steps.append(s)
                val_loss.append(float(row["val_loss"]))
    
    plt.figure(figsize=(8, 4.5))
    plt.plot(train_steps, train_loss, label="Training Loss", color="tab:blue", lw=2)
    if val_steps:
        plt.plot(val_steps, val_loss, label="Validation Loss", color="tab:orange", lw=2, linestyle="--", marker="o")
    plt.xlabel("Iteration Step", fontsize=11)
    plt.ylabel("Cross-Entropy Loss", fontsize=11)
    plt.title("Amharic GPT Training & Validation Loss Curves", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print(f"Metrics file not found at {metrics_file}. Training a quick 20-step toy demonstration...")
    from src.optimizer import AdamW
    from src.loss import cross_entropy_loss
    from src.data import AmharicDataset
    
    ds = AmharicDataset(char_tokens * 10, block_size=16)
    opt = AdamW(model.get_params_and_grads(), lr=1e-3)
    losses = []
    for i in range(20):
        X, Y = ds.get_batch(2)
        l, dl = cross_entropy_loss(model.forward(X, training=True), Y)
        model.zero_grad()
        model.backward(dl)
        opt.step()
        losses.append(l)
    plt.plot(losses, marker='o')
    plt.title("Live Toy Training Loss")
    plt.show()

## 5. Autoregressive Text Generation
Generates Amharic completions using incremental Key-Value (KV) caching, nucleus sampling (top-p), top-k filtering, and repetition penalty.

In [ ]:
# Load trained checkpoint if available
ckpt_dir = os.path.join(repo_root, "outputs", "small_amharic_run", "checkpoints")
if not os.path.exists(os.path.join(ckpt_dir, "checkpoint_best_meta.json")):
    ckpt_dir = os.path.join(repo_root, "checkpoints")

gen_model, gen_cfg, gen_tok, meta = load_checkpoint(ckpt_dir, name="checkpoint_best" if os.path.exists(os.path.join(ckpt_dir, "checkpoint_best_meta.json")) else "checkpoint")

prompt = "ሰላም"
print(f"Input Prompt: '{prompt}'\n")

# 1. Greedy Generation
out_greedy = generate(gen_model, gen_tok, prompt=prompt, max_new_tokens=40, greedy=True, use_cache=True)
print("[1. Greedy Decoding]:")
print(out_greedy)
print("-" * 50)

# 2. Nucleus (Top-p) + Top-k + Repetition Penalty Sampling
out_sample = generate(
    model=gen_model,
    tokenizer=gen_tok,
    prompt=prompt,
    max_new_tokens=40,
    temperature=0.8,
    top_k=20,
    top_p=0.9,
    repetition_penalty=1.1,
    greedy=False,
    use_cache=True,
    rng=np.random.RandomState(42),
)
print("[2. Nucleus (Top-p=0.9, Top-k=20, RepPenalty=1.1) with KV Cache]:")
print(out_sample)
print("-" * 50)